In [1]:
# Install if needed:
# !pip install pandas scikit-learn nltk matplotlib

import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

nltk.download("stopwords")
nltk.download("wordnet")

# 1. Load dataset
df = pd.read_csv("appleStore_description3.csv")

# Change this if your description column has another name
text_col = "description"

# 2. NLP preprocessing
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean(text):
    text = re.sub(r"[^a-zA-Z\s]", "", str(text).lower())
    words = [lemmatizer.lemmatize(w) for w in text.split()
             if w not in stop_words and len(w) > 2]
    return " ".join(words)

df = df.dropna(subset=[text_col])
df["clean_text"] = df[text_col].apply(clean)

# 3. TF-IDF
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df["clean_text"])

# 4. K-Means
k = 5
model = KMeans(n_clusters=k, random_state=42, n_init=10)
df["Cluster"] = model.fit_predict(X)

# 5. Show results
print(df[[text_col, "Cluster"]].head(10))
print("\nCluster counts:")
print(df["Cluster"].value_counts().sort_index())

# 6. Top words in each cluster
terms = tfidf.get_feature_names_out()

for i in range(k):
    words = model.cluster_centers_[i].argsort()[-10:][::-1]
    print(f"\nCluster {i}:",
          ", ".join(terms[j] for j in words))

# 7. Visualize clusters
pca = PCA(n_components=2)
points = pca.fit_transform(X.toarray())

plt.figure(figsize=(8, 5))
plt.scatter(points[:, 0], points[:, 1], c=df["Cluster"])
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Apple Store App Description - K-Means Clusters")
plt.show()

# 8. Save result
df.to_csv("appleStore_description3_clustered.csv", index=False)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


KeyError: ['description']